# GE-YOLOv8 Training on Kaggle
## Custom YOLOv8 with Gradient Search (CSP) and Efficient Channel Attention (ECA) modules

This notebook demonstrates training a custom GE-YOLOv8 model with architectural modifications including:
- **CSP (Cross Stage Partial)** - Gradient Search module
- **ECAAttention** - Efficient Channel Attention mechanism

### Environment Setup
The custom model requires installing the repository source code to access custom modules.

In [ ]:
# Step 1: Install base requirements
!pip install torch torchvision
!pip install ultralytics einops

In [ ]:
# Step 2: Clone the GE-YOLOv8 repository
import os

repo_url = "https://github.com/tunadev-io/Yolov8-GS-ECA.git"
repo_name = "Yolov8-GS-ECA"

if not os.path.exists(repo_name):
    !git clone {repo_url}
    print(f"✓ Repository cloned to {repo_name}")
else:
    print(f"✓ Repository {repo_name} already exists")

In [ ]:
# Step 3: Copy custom modules to ultralytics installation
import shutil
import ultralytics
import os
import re

# Get ultralytics installation path
ultralytics_path = os.path.dirname(ultralytics.__file__)
repo_path = repo_name

# Copy custom attention modules
src_modules = os.path.join(repo_path, 'nn', 'modules')
dst_modules = os.path.join(ultralytics_path, 'nn', 'modules')

# Copy Attention.py and CoordAttention.py
for module_file in ['Attention.py', 'CoordAttention.py']:
    src = os.path.join(src_modules, module_file)
    dst = os.path.join(dst_modules, module_file)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"✓ Copied {module_file}")

# Add CSP class to block.py
src_block = os.path.join(src_modules, 'block.py')
dst_block = os.path.join(dst_modules, 'block.py')

with open(src_block, 'r') as f:
    src_content = f.read()

# Extract CSP class definition
csp_match = re.search(r'(class CSP\(.*?\n(?:.*?\n)*?^    def forward.*?\n(?:.*?\n)*?^        return .*?\n)', 
                     src_content, re.MULTILINE)

if csp_match:
    with open(dst_block, 'r') as f:
        dst_content = f.read()
    
    if 'class CSP(' not in dst_content:
        csp_class = csp_match.group(1)
        with open(dst_block, 'a') as f:
            f.write('\n\n' + csp_class)
        print("✓ Added CSP class to block.py")
    else:
        print("✓ CSP already exists in block.py")

# Update __init__.py to export custom modules
init_py = os.path.join(dst_modules, '__init__.py')
with open(init_py, 'r') as f:
    init_content = f.read()

# Add imports for custom modules
if 'from .Attention import' not in init_content:
    init_content = init_content.replace(
        'from .transformer import (',
        'from .Attention import ECAAttention, GAM_Attention, ShuffleAttention, EMA\nfrom .CoordAttention import CoordAtt\nfrom .transformer import ('
    )

# Add CSP to block import
if ', CSP' not in init_content:
    init_content = init_content.replace('from .block import (', 'from .block import (\n    CSP,')

# Update __all__
custom_exports = ['CSP', 'ECAAttention', 'GAM_Attention', 'ShuffleAttention', 'EMA', 'CoordAtt']
for export in custom_exports:
    if f'"{export}"' not in init_content and f"'{export}'" not in init_content:
        init_content = init_content.replace('__all__ = (', f'__all__ = (\n    "{export}",', 1)

with open(init_py, 'w') as f:
    f.write(init_content)
print("✓ Updated __init__.py")

# Update tasks.py to import custom modules
tasks_py = os.path.join(ultralytics_path, 'nn', 'tasks.py')
with open(tasks_py, 'r') as f:
    tasks_content = f.read()

# Add CSP
if '\n    CSP,' not in tasks_content:
    tasks_content = tasks_content.replace('    C3x,\n', '    C3x,\n    CSP,\n')

# Add custom attention modules
custom_modules = ['ECAAttention', 'CoordAtt', 'ShuffleAttention', 'GAM_Attention', 'EMA']
for module in custom_modules:
    if f'\n    {module},' not in tasks_content:
        tasks_content = tasks_content.replace('    Detect,\n', f'    Detect,\n    {module},\n')

with open(tasks_py, 'w') as f:
    f.write(tasks_content)
print("✓ Updated tasks.py")

print("\n✅ Custom GE-YOLOv8 modules installed successfully!")
print("⚠️  Note: If you encounter errors, restart the kernel and continue from Step 4.")

In [ ]:
# Step 4: Verify custom modules are accessible
import sys
print(f"Python version: {sys.version}")
print(f"\nVerifying custom modules...")

try:
    from ultralytics.nn.modules.block import CSP
    from ultralytics.nn.modules.Attention import ECAAttention
    print("✓ CSP module imported successfully")
    print("✓ ECAAttention module imported successfully")
    print("\nCustom modules are ready for use!")
except ImportError as e:
    print(f"✗ Error importing modules: {e}")
    print("Please restart the kernel and try again.")

### Dataset Configuration
Generate the dataset configuration file for lumbar disc herniation grading dataset.

In [ ]:
# Step 5: Change to repository directory for model access
os.chdir(repo_name)
print(f"✓ Changed directory to {os.getcwd()}")

In [ ]:
# Step 6: Generate lumbar_data.yaml configuration file
import yaml

# Dataset configuration for lumbar disc herniation grading
# 10 classes representing different herniation grades
dataset_config = {
    'path': '/kaggle/input/lumbar-dataset',  # Update this path to your Kaggle dataset
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',  # optional
    
    # Lumbar disc herniation grading classes
    'nc': 10,  # number of classes
    'names': [
        '1A',   # Grade 1A
        '1B',   # Grade 1B
        '1C',   # Grade 1C
        '2A',   # Grade 2A
        '2B',   # Grade 2B
        '2C',   # Grade 2C
        '2AB',  # Grade 2AB
        '3A',   # Grade 3A
        '3B',   # Grade 3B
        '3AB'   # Grade 3AB
    ]
}

# Save configuration file
config_path = 'lumbar_data.yaml'
with open(config_path, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False, sort_keys=False)

print(f"✓ Dataset configuration saved to: {config_path}")
print("\nDataset configuration:")
with open(config_path, 'r') as f:
    print(f.read())

### Model Configuration
The custom model uses `yolov8-ECA-CSP.yaml` which combines:
- **CSP blocks** in the backbone for gradient search
- **ECAAttention** in the head for efficient channel attention

In [ ]:
# Step 7: Verify the model configuration file exists
model_config = 'models/v8/yolov8-ECA-CSP.yaml'
if os.path.exists(model_config):
    print(f"✓ Model configuration found: {model_config}")
    
    # Display the model architecture
    print("\nModel architecture preview:")
    with open(model_config, 'r') as f:
        print(f.read())
else:
    print(f"✗ Model configuration not found: {model_config}")

### Training
Initialize and train the GE-YOLOv8 model with custom architecture.

In [ ]:
# Step 8: Initialize the custom GE-YOLOv8 model
from ultralytics import YOLO

# Load the custom model architecture
model = YOLO('models/v8/yolov8-ECA-CSP.yaml')
print("✓ Custom GE-YOLOv8 model initialized")
print(f"\nModel architecture: YOLOv8 with CSP (Gradient Search) + ECA (Efficient Channel Attention)")

In [ ]:
# Step 9: Train the model
# Training parameters
results = model.train(
    data='lumbar_data.yaml',      # Dataset configuration
    epochs=100,                    # Number of training epochs
    imgsz=640,                     # Input image size
    batch=16,                      # Batch size
    name='ge-yolov8-lumbar',       # Experiment name
    project='runs/detect',         # Project directory
    patience=50,                   # Early stopping patience
    save=True,                     # Save checkpoints
    device=0,                      # GPU device (0 for first GPU, 'cpu' for CPU)
    workers=8,                     # Number of dataloader workers
    exist_ok=True,                 # Overwrite existing project
)

print("\n✓ Training completed!")
print(f"Results saved to: {results.save_dir}")

### Validation
Evaluate the trained model on the validation set.

In [ ]:
# Step 10: Validate the model
metrics = model.val()

print("\n=== Validation Metrics ===")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

### Model Export (Optional)
Export the trained model to different formats for deployment.

In [ ]:
# Step 11: Export the model (optional)
# Uncomment to export the model

# Export to ONNX format
# model.export(format='onnx', dynamic=True, simplify=True)

# Export to TensorRT
# model.export(format='engine', device=0, half=True)

# Export to TorchScript
# model.export(format='torchscript')

print("\n✓ Model export completed (if enabled)")

### Notes

**Custom Modules:**
- `CSP` (Gradient Search): Located in `nn/modules/block.py`
- `ECAAttention`: Located in `nn/modules/Attention.py`

**Dataset Structure:**
```
/kaggle/input/lumbar-dataset/
├── images/
│   ├── train/
│   ├── val/
│   └── test/
└── labels/
    ├── train/
    ├── val/
    └── test/
```

**Training Tips:**
1. Adjust `batch` size based on available GPU memory
2. Modify `epochs` for longer/shorter training
3. Update dataset paths in `lumbar_data.yaml` to match your Kaggle input
4. Monitor training metrics in TensorBoard: `%load_ext tensorboard` and `%tensorboard --logdir runs/detect`

**Model Architecture:**
- The `yolov8-ECA-CSP.yaml` configuration specifies 10 classes (`nc: 10`)
- CSP blocks replace C2f blocks in the backbone for improved gradient flow
- ECAAttention is applied to the P5 head output for enhanced feature representation

**Important:** If you encounter import errors after Step 3, please restart the kernel and continue from Step 4.